In [1]:
import pandas as pd
import cv2

# Ler os dados do CSV
csv_file = '1_1_Device 1.csv'
df = pd.read_csv(csv_file)

# Ordenar os dados por timestamp para garantir a ordem correta
df = df.sort_values(by='timestamp').reset_index(drop=True)

# Carregar o vídeo
video_file = 'data/video.mp4'
cap = cv2.VideoCapture(video_file)

if not cap.isOpened():
    print("Erro ao abrir o vídeo.")
    exit()

# Informações do vídeo
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"FPS: {fps}, Width: {width}, Height: {height}, Total Frames: {total_frames}")

# Preparar o writer para salvar o novo vídeo
output_file = 'output_video.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec para salvar em formato MP4
out = cv2.VideoWriter(output_file, fourcc, fps, (width, height))

if not out.isOpened():
    print("Erro ao abrir o vídeo de saída.")
    cap.release()
    exit()

# Função para encontrar o label correto para um dado frame
def get_label_for_frame(timestamp):
    if timestamp < df['timestamp'].iloc[0]:
        return df['left_label'].iloc[0]
    for i in range(len(df) - 1):
        if df['timestamp'].iloc[i] <= timestamp < df['timestamp'].iloc[i + 1]:
            return df['left_label'].iloc[i]
    return df['left_label'].iloc[-1]

# Processar cada frame do vídeo
frame_index = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Calcular timestamp do frame atual
    timestamp = frame_index / fps * 1000  # Convertendo para milissegundos

    # Obter o texto correspondente ao timestamp atual
    text = get_label_for_frame(timestamp)

    # Adicionar o texto ao frame
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    color = (255, 255, 255)
    thickness = 2
    text_size, _ = cv2.getTextSize(text, font, font_scale, thickness)
    text_x = (frame.shape[1] - text_size[0]) // 2
    text_y = (frame.shape[0] + text_size[1]) // 2
    cv2.putText(frame, text, (text_x, text_y), font, font_scale, color, thickness)

    # Escrever o frame modificado no novo vídeo
    out.write(frame)
    frame_index += 1

# Liberar os recursos
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Vídeo salvo como {output_file}")


FileNotFoundError: [Errno 2] No such file or directory: 'data/Accelerometer_Anotador.csv'